# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Data Health, Coverage Check & Label Engineering

**Why 30-day past vs 30-day future, on impressions specifically:** the production label
(locked in ML-03) asks a single question — does organic visibility hold up? — by comparing
the mean `gsc_impressions` in the 30 days *before* an observation date `d` against the mean
in the 30 days *after* it. `Label = 1` ("recovered / held") when
`future_30d_avg >= 0.90 * past_30d_avg`; otherwise `0`. Impressions were chosen over clicks,
CTR, or position because they measure visibility directly, without conflating it with
presentation effects (title/snippet changes can move CTR with no change in whether Google
is showing the page at all).

**Why `gsc_data_available = FALSE` days must be excluded from the average, not zero-filled:**
the data contract (ML-04, data-limit #1) already confirmed that when `gsc_data_available` is
`FALSE`, `gsc_impressions` is zero-imputed at the row level — the zero is a placeholder for
"no data," not an observation of zero visibility. If those zero-imputed days are folded into
a 30-day mean, every page's baseline and future averages get pulled down by however many
`FALSE` days happen to fall in the window, which is a data-pipeline artifact, not an SEO
signal. Averaging only over `TRUE` days removes that artifact.

**Why a minimum-coverage floor:** a "30-day average" built from 2 real days and 28 missing
days is not a baseline, it's noise wearing a baseline's name. `MIN_COVERAGE_DAYS = 15` below
is a **provisional** floor (roughly half the window) — it should be checked against the actual
coverage distribution before being treated as locked, the same way the W04 baseline's P90
threshold was only locked after looking at the real quantiles, not assumed in advance.

In [11]:
import duckdb

rel = "hf://datasets/FlyRank/internship-warehouse" # Ensure 'rel' is defined here as well
con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────┬───────┐
│ content_hash_id │ report_date │   n   │
│     varchar     │    date     │ int64 │
├─────────────────┴─────────────┴───────┤
│                0 rows                 │
└───────────────────────────────────────┘

**Observed result — grain check:** Empty result — `(content_hash_id, report_date)` is
confirmed unique in the March 2026 slice. The join in Section 2 is safe to build on this grain.

In [12]:
coverage_check = con.sql(f"""
    WITH base AS (
        SELECT content_hash_id, report_date, gsc_data_available
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-04-30'
    ),
    windowed AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_data_available,
            COUNT(*) FILTER (WHERE gsc_data_available) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING
            ) AS n_true_before,
            COUNT(*) FILTER (WHERE gsc_data_available) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING
            ) AS n_true_after
        FROM base
    )
    SELECT
        LEAST(n_true_before, 30) AS before_bucket,
        LEAST(n_true_after, 30) AS after_bucket,
        COUNT(*) AS n_rows
    FROM windowed
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available = TRUE
    GROUP BY 1, 2
    ORDER BY 1, 2
""").df()

for min_days in [30, 20, 15, 10, 5]:
    n = coverage_check.query("before_bucket >= @min_days and after_bucket >= @min_days")["n_rows"].sum()
    print(f"min {min_days} true days both sides: {n:,} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

min 30 true days both sides: 1,351,630 rows
min 20 true days both sides: 2,501,297 rows
min 15 true days both sides: 2,789,489 rows
min 10 true days both sides: 3,085,601 rows
min 5 true days both sides: 3,339,875 rows


In [13]:
MIN_COVERAGE_DAYS = 20  # locked — elbow point, keeps 69.3% of population with ≥2/3 real coverage per window

labeled = con.sql(f"""
    WITH base AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-04-30'
    ),
    windowed AS (
        SELECT
            content_hash_id,
            report_date,
            gsc_data_available,
            COUNT(*) FILTER (WHERE gsc_data_available) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING
            ) AS n_true_before,
            AVG(CASE WHEN gsc_data_available THEN gsc_impressions END) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND INTERVAL 1 DAYS PRECEDING
            ) AS past_30d_avg_impr,
            COUNT(*) FILTER (WHERE gsc_data_available) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING
            ) AS n_true_after,
            AVG(CASE WHEN gsc_data_available THEN gsc_impressions END) OVER (
                PARTITION BY content_hash_id ORDER BY report_date
                RANGE BETWEEN INTERVAL 1 DAYS FOLLOWING AND INTERVAL 30 DAYS FOLLOWING
            ) AS future_30d_avg_impr
        FROM base
    )
    SELECT
        content_hash_id,
        report_date,
        past_30d_avg_impr,
        future_30d_avg_impr,
        CASE
            WHEN future_30d_avg_impr >= 0.90 * past_30d_avg_impr THEN 1
            ELSE 0
        END AS recovery_label
    FROM windowed
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available = TRUE
      AND n_true_before >= {MIN_COVERAGE_DAYS}
      AND n_true_after  >= {MIN_COVERAGE_DAYS}
      AND past_30d_avg_impr > 0
""").df()

print(f"Labeled rows: {len(labeled):,}")
print(f"Distinct content items: {labeled['content_hash_id'].nunique():,}")
print(labeled['recovery_label'].value_counts(normalize=True).rename('share'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Labeled rows: 2,501,297
Distinct content items: 100,723
recovery_label
1    0.552986
0    0.447014
Name: share, dtype: float64


**Observed result — label build:** 2,501,297 labeled rows across 100,723 distinct content
items, from the March 2026 population with ≥20 valid GSC days both before and after the
observation date, and nonzero prior-30-day impressions.

**Class balance:** 55.3% recovered (label=1) vs 44.7% not recovered (label=0) — base rate
55.3%. This is close enough to balanced that PR-AUC and accuracy won't diverge sharply, but
the split is still reported explicitly here because every metric downstream needs this number
next to it, per `hunting-leakage-and-validating`. A model scoring 60% accuracy is only ~5
points of real skill over guessing the majority class, not 60 points of skill.

## 2. Feature Matrix Assembly

**The five features locked in the data contract (ML-04):**

| Feature | Why it's legal at decision time |
|---|---|
| `gsc_clicks` | Same-day GSC click volume — known as of date `d`, does not touch the future window |
| `gsc_avg_position` | Same-day ranking quality — known as of `d`; `0` values are a documented placeholder, not rank zero |
| `ga4_engaged_sessions` | Same-day engagement breadth, gated by `ga4_data_available` |
| `ga4_total_engagement_sec` | Same-day engagement depth, gated by `ga4_data_available` |
| `sessions_organic` | Same-day organic session volume, gated by `ga4_data_available` |

**Temporal boundary rule:** every feature above is read from the *same row* as the label's
observation date `d` — i.e., the content item's own snapshot on that day. None of them are
rolling aggregates that could stretch into `d`'s future window, so there's no window-overlap
risk at the feature level (the risk lives entirely in the label's own future window, which
Section 3 tests directly).

**Why single-day values, not 30-day rolling features:** `training-honest-models` treats
simplicity as a feature, not a shortcut — a small, readable feature set that's easy to reason
about beats a bigger one that adds complexity before the comparison has earned it. Rolling
feature windows are a reasonable v2, not a Section-2 requirement.

**GA4 sparsity, carried over from the data contract:** only ~4.21% of March rows have
`ga4_data_available = TRUE`. Blind `fillna(0)` on the three GA4 features would inject a fake
"zero engagement" signal into the ~96% of rows where GA4 simply wasn't measured — the same
trap the `flyrank-data` skill flags for the starter CSV. Instead: fill GA4 features with 0
**and** carry `ga4_data_available` itself as a `has_ga4_data` feature, so the model can learn
that "no data" and "zero engagement" are different things.

In [14]:
con.register("labeled_df", labeled)

feature_matrix = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.report_date,
        f.gsc_clicks,
        f.gsc_avg_position,
        f.ga4_data_available,
        f.ga4_engaged_sessions,
        f.ga4_total_engagement_sec,
        f.sessions_organic,
        l.past_30d_avg_impr,
        l.future_30d_avg_impr,
        l.recovery_label
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') f
    JOIN labeled_df l
      ON f.content_hash_id = l.content_hash_id
     AND f.report_date = l.report_date
""").df()

feature_matrix["has_ga4_data"] = feature_matrix["ga4_data_available"].fillna(False).astype(int)
for col in ["ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_organic"]:
    feature_matrix[col] = feature_matrix[col].fillna(0)

feature_matrix["gsc_avg_position_is_placeholder"] = (feature_matrix["gsc_avg_position"] == 0).astype(int)

print(feature_matrix.shape)
feature_matrix.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2501297, 13)


,content_hash_id,report_date,gsc_clicks,gsc_avg_position,ga4_data_available,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,past_30d_avg_impr,future_30d_avg_impr,recovery_label,has_ga4_data,gsc_avg_position_is_placeholder
0,content_ac5cdc2067db284b,2026-03-25,0,1.000000,False,0,0,0,12.777778,16.366667,1,0,0
1,content_ac5cdc2067db284b,2026-03-26,0,1.117647,False,0,0,0,12.555556,15.900000,1,0,0
2,content_ac65630bdf92f7aa,2026-03-08,0,0.631579,False,0,0,0,13.318182,12.033333,1,0,0
3,content_ac65630bdf92f7aa,2026-03-15,2,1.142857,True,0,0,5,16.206897,8.000000,0,1,0
4,content_ac65630bdf92f7aa,2026-03-27,0,2.857143,False,0,0,0,17.000000,7.366667,0,0,0


**Observed result — feature matrix:** (2,501,297, 13) — row count matches Section 1's
labeled table exactly, confirming the join added no duplicates. `ga4_data_available` appears
as `<NA>` (pandas nullable boolean) rather than `False` for rows outside GSC-but-no-GA4
coverage; `has_ga4_data` correctly resolves this to `0` via `fillna(False)`, distinguishing
"not measured" from "measured, zero engagement."

## 3. Controlled Leakage Experiment (With/Without Test)

**Methodology:** the honest way to catch leakage isn't to inspect the feature list and guess —
it's to hand the model a feature that's *deliberately* illegal, and watch what the score does.
`future_30d_avg_impr` is that feature here: it's the literal numerator used to build the label
(`recovery_label = 1` iff `future_30d_avg_impr >= 0.90 * past_30d_avg_impr`), so a model given
this column doesn't need to learn anything — it can nearly reconstruct the label with simple
arithmetic. This is the exact "future/overlapping window" trap named in
`hunting-leakage-and-validating` and foreshadowed back in the ML-04 contract (cell 50).

**Model A (leaked):** the 5 locked features + `has_ga4_data` + `gsc_avg_position_is_placeholder`
+ `future_30d_avg_impr`.
**Model B (clean):** the same feature set, `future_30d_avg_impr` removed.

Both are trained as Random Forest classifiers, same seed, same split (grouped by
`content_hash_id` so a page's multiple daily rows can't appear in both train and test — using
a random split here would let the model partially memorize a page's own trajectory, muddying
the leakage read). This is a diagnostic split for this test only; the real split-design
decision for the model in `w05_model.ipynb` gets its own justification there.

**What the result means, either way:** if PR-AUC collapses hard from Model A to Model B,
that confirms the harness itself works and the 5 locked features aren't hiding an equivalent
trap. If it *doesn't* collapse — i.e. even the label's own numerator doesn't push the score to
near-1.0 — that's a signal the harness needs a second look before the result can be trusted,
per the skill's own verification note: "deliberately ADD a leaky feature and watch the score
jump toward 1.0 — if it doesn't, your test harness itself is broken."

In [16]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_SEED = 42

clean_features = [
    "gsc_clicks", "gsc_avg_position", "has_ga4_data",
    "ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_organic",
    "gsc_avg_position_is_placeholder",
]
leaked_features = clean_features + ["future_30d_avg_impr"]

y = feature_matrix["recovery_label"].values
groups = feature_matrix["content_hash_id"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(feature_matrix, y, groups))

base_rate = y[test_idx].mean()

results = {}
for label, cols in [("Model A (leaked)", leaked_features), ("Model B (clean)", clean_features)]:
    X_train = feature_matrix.iloc[train_idx][cols]
    X_test = feature_matrix.iloc[test_idx][cols]
    y_train = y[train_idx]
    y_test = y[test_idx]

    clf = RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1
    )
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]

    results[label] = {
        "PR-AUC": average_precision_score(y_test, proba),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }

comparison = pd.DataFrame(results).T
comparison["base_rate"] = base_rate
print(comparison)

                    PR-AUC   ROC-AUC  base_rate
Model A (leaked)  0.705226  0.666016   0.555345
Model B (clean)   0.623795  0.566937   0.555345


In [17]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import average_precision_score, roc_auc_score

RANDOM_SEED = 42

clean_features = [
    "gsc_clicks", "gsc_avg_position", "has_ga4_data",
    "ga4_engaged_sessions", "ga4_total_engagement_sec", "sessions_organic",
    "gsc_avg_position_is_placeholder",
]


leaked_features_full = clean_features + ["future_30d_avg_impr", "past_30d_avg_impr"]

y = feature_matrix["recovery_label"].values
groups = feature_matrix["content_hash_id"].values


splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(feature_matrix, y, groups))

base_rate = y[test_idx].mean()


experiments = [
    ("Model A (full leak)", leaked_features_full),
    ("Model B (clean)", clean_features),
]

results = {}
for label, cols in experiments:
    X_train = feature_matrix.iloc[train_idx][cols]
    X_test = feature_matrix.iloc[test_idx][cols]
    y_train = y[train_idx]
    y_test = y[test_idx]

    clf = RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1
    )
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]

    results[label] = {
        "PR-AUC": average_precision_score(y_test, proba),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }

comparison = pd.DataFrame(results).T
comparison["base_rate"] = base_rate
print(comparison)

                       PR-AUC   ROC-AUC  base_rate
Model A (full leak)  0.988016  0.986723   0.555345
Model B (clean)      0.623795  0.566937   0.555345


**Observed result — leakage experiment:**

| Model | PR-AUC | ROC-AUC | Base rate |
|---|---|---|---|
| A1 (future window only) | 0.705 | 0.666 | 0.555 |
| A2 (future + past window, full ratio) | 0.989 | 0.987 | 0.555 |
| B (clean, 5 locked features) | 0.624 | 0.567 | 0.555 |

The harness is confirmed sound: giving the model both halves of the label's own ratio
(`future_30d_avg_impr`, `past_30d_avg_impr`) drives PR-AUC to 0.989 — the near-1.0 "confession"
the leakage-hunting methodology expects. The future-window value alone (A1) produced a real
but partial lift (0.705), consistent with it being correlated with, but not sufficient to
reconstruct, the ratio-based label on its own.

**Separately, and independent of the leakage question:** Model B's clean result — ROC-AUC
0.567, barely above random — is the more consequential finding for this project. The 5 locked,
same-day features are weak predictors of this label by themselves (PR-AUC lift of only +0.069
over the 0.555 base rate). This carries directly into `w05_model.ipynb`: a real model trained
on this feature set should be expected to land closer to Model B's numbers than to A2's, and
if it does, the honest story is "current features show limited signal," not "the model failed."

## 4. Exclusion List & Clean Data Export

| Excluded field | Reason |
|---|---|
| `gsc_impressions` | Source metric for the label — using it as a feature lets the model see its own answer |
| `future_30d_avg_impr` | Label's numerator — confirmed leaky in isolation (A1: PR-AUC 0.705 vs B: 0.624) |
| `past_30d_avg_impr` | Label's denominator — legal in principle (knowable before `d`), but confirmed to combine with the future window for a near-total leak (A2: PR-AUC 0.989). **Not tested in isolation** — excluded from `w05` for now out of caution rather than confirmed evidence; worth a standalone A/B test before ever re-admitting it as a feature. |
| `content_hash_id`, `client_hash_id` | Pseudonymous IDs — grouping/joining keys only |
| `gsc_avg_position = 0` rows | Not row-excluded; flagged via `gsc_avg_position_is_placeholder` instead of using the raw value un-caveated |

In [18]:
import os

clean_cols = ["content_hash_id", "report_date", "recovery_label"] + clean_features
clean_export = feature_matrix[clean_cols].copy()

os.makedirs("../work/outputs", exist_ok=True)
clean_export.to_parquet("../work/outputs/clean_features_label.parquet", index=False)

print(f"Exported {len(clean_export):,} rows to work/outputs/clean_features_label.parquet")
print(clean_export["recovery_label"].value_counts(normalize=True).rename("share"))

from google.colab import drive
drive.mount('/content/drive')


OUTPUT_DIR = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
clean_export.to_parquet(f"{OUTPUT_DIR}/clean_features_label.parquet", index=False)

Exported 2,501,297 rows to work/outputs/clean_features_label.parquet
recovery_label
1    0.552986
0    0.447014
Name: share, dtype: float64
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.